# Бенчмарк: дообученная my-coder vs исходная qwen2.5-coder:7b

Две проверки на **той же GGUF Q4_K_M**, что пойдёт на Mac:
1. **Объяснение кода** — 18 отложенных вопросов (нет в обучении), с системным промптом и без → `eval_gguf.md`.
2. **Агент loca** — hard-задачи `bench/run.py`: доходит ли план до конца (узкое место №2).
3. **Пробник** `bench/probe.py`: задание перед работой (№8) и «сначала запусти» (№4).

Всё сводится в `runs/summary.txt`.

**Перед запуском** (панель справа):
- Settings: **GPU T4 x1**, **Internet On**
- **Add Input → Your Work → Datasets → `loca`** — **обновите** его свежим `loca.zip` из `Developer/coder` (New Version → Upload): в нём `bench/probe.py`
- **Add Input → Your Work → Notebooks → ноутбук дообучения v2** — подтянет его Output с `my-coder-Q4_K_M.gguf`

Запуск: **Save Version → Save & Run All** (≈1–1,5 часа). Пришлите `eval_gguf.md` и `runs/summary.txt` из Output.

In [ ]:
# 1. Настройки
MODELS   = ["my-coder", "qwen2.5-coder:7b"]   # дообученная и исходная
CTX      = 16384
TASKS    = "hard"     # "hard", "core" или "" (все)
TIMEOUT  = 900
RUN_EXPLAIN = True
RUN_LOCA    = True
RUN_PROBE   = True   # прицельные проверки узких мест №8 и №4 (bench/probe.py), ~5 мин на модель
BASELINE = "runs/hard3.json"   # qwen3.5:4b с мака от 19.09 — для сравнения

In [ ]:
# 2. Копируем loca из Input в рабочую папку (Input только для чтения)
import glob, os, shutil, subprocess
src = glob.glob("/kaggle/input/**/bench/run.py", recursive=True)
assert src, "Не найден loca: загрузите loca.zip через Add Input → Upload"
repo_src = os.path.dirname(os.path.dirname(src[0]))
REPO = "/kaggle/working/loca"
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(repo_src, REPO)
os.chdir(REPO)
print("loca:", REPO); print(os.listdir(REPO))

In [ ]:
# 3. Ставим и запускаем Ollama (2–3 минуты)
!apt-get -qq update >/dev/null && apt-get -qq install -y zstd pciutils lshw >/dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import time, urllib.request
env = {**os.environ, "OLLAMA_FLASH_ATTENTION": "1", "OLLAMA_KV_CACHE_TYPE": "q8_0",
       "OLLAMA_KEEP_ALIVE": "60m"}
log = open("/kaggle/working/ollama.log", "w")
ollama = subprocess.Popen(["ollama", "serve"], env=env, stdout=log, stderr=log)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2); print("Ollama запущена"); break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("Ollama не поднялась, см. /kaggle/working/ollama.log")

In [ ]:
# 4. Модели: исходная из библиотеки Ollama + своя GGUF с тем же шаблоном чата (нужен для tool calling в loca)
import re, glob, subprocess
!ollama pull qwen2.5-coder:7b
gg = sorted(glob.glob("/kaggle/input/**/*.gguf", recursive=True))
print("GGUF:", gg)
assert gg, "Нет GGUF: добавьте Output ноутбука дообучения через Add Input → Your Work → Notebooks"
base_mf = subprocess.check_output(["ollama", "show", "qwen2.5-coder:7b", "--modelfile"], text=True)
tmpl = re.search(r'TEMPLATE """(.*?)"""', base_mf, re.S).group(1)
params = "\n".join(l for l in base_mf.splitlines() if l.startswith("PARAMETER"))
open("/kaggle/working/Modelfile", "w").write(f'FROM {gg[0]}\nTEMPLATE """{tmpl}"""\n{params}\n')
!ollama create my-coder -f /kaggle/working/Modelfile
!ollama list


In [ ]:
# 5. Объяснение кода: 18 отложенных вопросов, обе модели, с промптом и без
import json, urllib.request
SYSTEM_PROMPT = """Ты — опытный преподаватель программирования. Когда тебя просят объяснить код:
1. Одной-двумя фразами скажи, что код делает в целом.
2. Назови механизм языка, на котором держится его поведение (например: замыкание и позднее связывание, изменяемый объект и общие ссылки, вычисление значения по умолчанию при определении функции, короткое вычисление and/or, ленивость генераторов), и объясни, ПОЧЕМУ код работает именно так — шаг за шагом, как это выполняет интерпретатор.
3. Если в коде есть подвох или неочевидное поведение — назови его прямо и покажи, когда он проявится. Если подвоха нет — так и скажи, не выдумывай.
4. Укажи важные ограничения и граничные случаи: типы аргументов, хешируемость, порядок элементов, сложность.
5. Если есть ошибка или плохая практика — покажи исправленный вариант кода.
Не пересказывай код построчно, если это не помогает понять суть. Пиши по-русски, точно и без воды."""
EVAL = [
 {
  "id": "e01",
  "q": "Объясни, что делает этот код и почему он работает:\n\n```python\ndef f(xs):\n    seen = set()\n    return [x for x in xs if not (x in seen or seen.add(x))]\n```"
 },
 {
  "id": "e02",
  "q": "Почему этот код ведёт себя неожиданно при повторных вызовах?\n\n```python\ndef add(item, items=[]):\n    items.append(item)\n    return items\n```"
 },
 {
  "id": "e03",
  "q": "Объясни, что выведет код и почему:\n\n```python\nfuncs = [lambda: i for i in range(3)]\nprint([f() for f in funcs])\n```"
 },
 {
  "id": "e04",
  "q": "Объясни простыми словами, как работает этот декоратор:\n\n```python\nimport functools\ndef cache(fn):\n    memo = {}\n    @functools.wraps(fn)\n    def wrapper(*args):\n        if args not in memo:\n            memo[args] = fn(*args)\n        return memo[args]\n    return wrapper\n```"
 },
 {
  "id": "e05",
  "q": "Что выведет этот код? Объясни.\n\n```python\ngrid = [[0] * 3] * 3\ngrid[0][0] = 1\nprint(grid)\n```"
 },
 {
  "id": "e06",
  "q": "Почему тут ошибка, ведь x объявлен выше?\n\n```python\nx = 10\ndef show():\n    print(x)\n    x = 5\nshow()\n```"
 },
 {
  "id": "e07",
  "q": "Почему второй print выводит 0?\n\n```python\nsquares = (n * n for n in range(4))\nprint(sum(squares))\nprint(sum(squares))\n```"
 },
 {
  "id": "e08",
  "q": "Почему результат False? Как правильно сравнивать?\n\n```python\nprint(0.1 + 0.2 == 0.3)\n```"
 },
 {
  "id": "e09",
  "q": "Что выведет этот код и почему? Как исправить?\n\n```javascript\nfor (var i = 0; i < 3; i++) {\n  setTimeout(() => console.log(i), 0);\n}\n```"
 },
 {
  "id": "e10",
  "q": "Почему этот запрос ничего не возвращает, хотя клиенты без заказов есть?\n\n```sql\nCREATE TABLE clients(id INTEGER, name TEXT);\nCREATE TABLE orders(id INTEGER, client_id INTEGER);\nINSERT INTO clients VALUES (1,'Аня'),(2,'Боря'),(3,'Вика');\nINSERT INTO orders VALUES (10,1),(11,NULL);\nSELECT name FROM clients WHERE id NOT IN (SELECT client_id FROM orders);\n```"
 },
 {
  "id": "e11",
  "q": "Объясни, как работает эта функция и какая у неё сложность:\n\n```python\ndef find(arr, target):\n    lo, hi = 0, len(arr) - 1\n    while lo <= hi:\n        mid = (lo + hi) // 2\n        if arr[mid] == target:\n            return mid\n        if arr[mid] < target:\n            lo = mid + 1\n        else:\n            hi = mid - 1\n    return -1\n```"
 },
 {
  "id": "e12",
  "q": "Объясни, как устроена и как работает эта программа целиком:\n\n```python\nimport json\n\nclass Library:\n    def __init__(self, path):\n        self.path = path\n        self.books = self._load()\n\n    def _load(self):\n        try:\n            with open(self.path, encoding='utf-8') as f:\n                return json.load(f)\n        except FileNotFoundError:\n            return {}\n\n    def _save(self):\n        with open(self.path, 'w', encoding='utf-8') as f:\n            json.dump(self.books, f, ensure_ascii=False, indent=2)\n\n    def add(self, isbn, title):\n        self.books[isbn] = {'title': title, 'taken_by': None}\n        self._save()\n\n    def take(self, isbn, reader):\n        book = self.books.get(isbn)\n        if book is None or book['taken_by']:\n            return False\n        book['taken_by'] = reader\n        self._save()\n        return True\n\n    def give_back(self, isbn):\n        self.books[isbn]['taken_by'] = None\n        self._save()\n\nlib = Library('books.json')\nlib.add('111', 'Мастер и Маргарита')\nprint(lib.take('111', 'Петя'), lib.take('111', 'Маша'))\n\n```"
 },
 {
  "id": "e13",
  "q": "Почему выручка после merge получилась больше реальной? Объясни и исправь.\n\n```python\nimport pandas as pd\norders = pd.DataFrame({'order_id': [1, 2, 3], 'client': ['A', 'B', 'A'], 'amount': [100, 250, 80]})\ncontacts = pd.DataFrame({'client': ['A', 'A', 'B'], 'phone': ['111', '222', '333']})\ndf = orders.merge(contacts, on='client')\nprint(df['amount'].sum())\n```"
 },
 {
  "id": "e14",
  "q": "Сделай дашборд на Streamlit для продаж интернет-магазина. Данные в sales.csv: date, region, category, revenue, orders. Нужны фильтры по региону и периоду, KPI (выручка, заказы, средний чек), график выручки по месяцам и топ-5 категорий. Объясни, как он устроен."
 },
 {
  "id": "e15",
  "q": "Напиши SQL-запрос: выручка по месяцам и рост в процентах к прошлому месяцу. Таблица orders(id, user_id, created_at TEXT 'YYYY-MM-DD', amount). Объясни, как он работает."
 },
 {
  "id": "e16",
  "q": "Почему модель почти не учится, хотя loss считается без ошибок?\n\n```python\nimport torch\nimport torch.nn as nn\n\ntorch.manual_seed(0)\nmodel = nn.Sequential(nn.Linear(4, 3), nn.Softmax(dim=1))\nloss_fn = nn.CrossEntropyLoss()\nx = torch.randn(8, 4)\ny = torch.randint(0, 3, (8,))\nopt = torch.optim.SGD(model.parameters(), lr=0.5)\nfor step in range(3):\n    loss = loss_fn(model(x), y)\n    loss.backward()\n    opt.step()\n    print(round(loss.item(), 4))\n```"
 },
 {
  "id": "e17",
  "q": "Объясни простыми словами, что делает model.eval() и torch.no_grad() и почему на валидации нужны оба. Что будет, если забыть одно из них?"
 },
 {
  "id": "e18",
  "q": "Я дообучил модель LoRA и смёрджил адаптер. Как понять, какие слои изменились сильнее всего? Напиши код, который сравнивает два state_dict (до и после) и выводит слои по убыванию изменения, и объясни, какие метрики тут осмысленны."
 }
]

def chat(model, question, system=None):
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": question}]
    body = json.dumps({"model": model, "messages": msgs, "stream": False,
                       "options": {"temperature": 0, "num_ctx": 8192, "num_predict": 900}}).encode()
    req = urllib.request.Request("http://localhost:11434/api/chat", data=body, headers={"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(req, timeout=600))["message"]["content"].strip()

if RUN_EXPLAIN:
    with open("/kaggle/working/eval_gguf.md", "w", encoding="utf-8") as f:
        for e in EVAL:
            f.write(f"# {e['id']}\n\n{e['q']}\n\n")
            for m in MODELS:
                for tag, sysmsg in (("без промпта", None), ("с промптом", SYSTEM_PROMPT)):
                    f.write(f"## {m} — {tag}\n\n{chat(m, e['q'], sysmsg)}\n\n")
            f.flush()
            print("✓", e["id"])
    print("Готово: /kaggle/working/eval_gguf.md")

In [ ]:
# 6. Бенчмарк агента loca для каждой модели
os.makedirs("/kaggle/working/runs", exist_ok=True)
if RUN_LOCA:
    for m in MODELS:
        out = f"/kaggle/working/runs/kaggle_{m.replace(':', '_')}_{TASKS or 'all'}.json"
        cmd = f"python3 bench/run.py -m {m} --ctx {CTX} --timeout {TIMEOUT} --out {out}"
        if TASKS:
            cmd += f" --tasks {TASKS}"
        if BASELINE and os.path.exists(BASELINE):
            cmd += f" --baseline {BASELINE}"
        print("\n$", cmd)
        !{cmd} 2>&1 | tee -a /kaggle/working/runs/bench_log.txt

In [ ]:
# 6a. Узкие места №8 (задание) и №4 («сначала запусти») — прицельно, на каждой модели
os.makedirs("/kaggle/working/runs", exist_ok=True)
if RUN_PROBE:
    for m in MODELS:
        out = f"/kaggle/working/runs/probe_{m.replace(':', '_')}.json"
        cmd = f"python3 bench/probe.py -m {m} --ctx {CTX} --timeout 600 --out {out}"
        print("\n$", cmd)
        !{cmd} 2>&1 | tee -a /kaggle/working/runs/probe_log.txt


In [ ]:
# 7. Сводка: агент (№2 — доходит ли план до конца) + пробник (№8, №4)
lines = []
for p in sorted(glob.glob("/kaggle/working/runs/kaggle_*.json")):
    d = json.load(open(p))
    rs = d["results"]
    ok = sum(r["passed"] for r in rs)
    part = sum(r.get("partial", 0) for r in rs); tot = sum(r.get("partial_total", 0) for r in rs)
    errs = sum(r.get("tool_errors", 0) for r in rs); steps = sum(r.get("steps", 0) for r in rs)
    lines.append(f"{d['model']:<22} задач {ok}/{len(rs)} · целей {part}/{tot} · шагов {steps} · ошибок инструментов {errs}")
    for r in rs:
        nodes = r.get("plan_nodes") or []
        done = sum(n.get("status") == "done" for n in nodes)
        ps = r.get("plan_stats") or {}
        plan = (f"план {done}/{len(nodes)} done · откатов {ps.get('rollbacks', 0)} · "
                f"уже-сделано {ps.get('already_done', 0)}") if nodes else "без плана"
        lines.append(f"    {r['name']:<24} {'✓' if r['passed'] else '✗'}  цели {r.get('partial',0)}/{r.get('partial_total',0)}"
                     f"  шагов {r.get('steps',0)}  токенов {r.get('tokens_in',0)}  {plan}")
for p in sorted(glob.glob("/kaggle/working/runs/probe_*.json")):
    d = json.load(open(p))
    lines.append(f"\nпробник {os.path.basename(p)}:")
    for key, title in (("spec", "№8 задание"), ("verify", "№4 сначала запусти")):
        rs = d.get(key, [])
        lines.append(f"  {title}: {sum(r['ok'] for r in rs)}/{len(rs)}")
        for r in rs:
            extra = (f"выдумано={r.get('invented_paths') or '—'} критерий={r.get('runnable_done')} спрашивает={r.get('asks')}"
                     if key == "spec" else f"запустил={r.get('ran_code')} вывод={r.get('found')}")
            lines.append(f"    {r['id']:<12} {'✓' if r['ok'] else '✗'}  {extra}")
open("/kaggle/working/runs/summary.txt", "w").write("\n".join(lines))
print("\n".join(lines))
